# MosAIc Checkpoint 4: validating the Tree of Thought design

MosAIc forecasts whether a large US stock will fall a lot, stay roughly flat or rise a lot over the next 21 trading days, and explains the forecast with a ledger: one cited line per piece of evidence, each shifting the three probabilities.

Checkpoint 4 replaced the single-pass synthesis step with a Tree of Thought (ToT) search over possible readings of the evidence. This notebook builds only that slice of the agent, like the Checkpoint 3 notebook did for retrieval, and tests the design decisions on real filings and prices:

1. **Does searching over readings help?** Same evidence, four ways of turning it into a forecast: prompt only, single pass, greedy ToT (beam 1) and the full design (beam 3).
2. **Can "no effect" win against standard wording?** The Apple legal-proceedings case from Checkpoint 3, shown as a tree.
3. **Are beam width 3 and the 0.3 pruning margin sensible?** A sweep that costs no extra model calls.
4. **Search error or scoring error?** Does the beam find the ledger the scoring likes best, and does the scoring like the ledgers that turned out right?
5. **Does it fit the budget?** Model calls and time per forecast.
6. **Where does it help?** The same comparison in five test scenarios: right after results, mid-quarter, just before results, after other company events, and random weeks.

**What this notebook cannot test yet.** The main metric in the design is log loss against the probabilities implied by option prices, and that data is not bought yet. Here every forecast is scored against what actually happened (Brier score). With about 120 forecasts that is an indication, not a proof. The scoring weights stay at their defaults, since tuning them needs the option data.

**Before you run:** set `USER_AGENT` to your name and email (the SEC requires it). Use a GPU runtime (Runtime → Change runtime type → T4 GPU).

## 1. How the design answers the paper's four questions

Yao et al. (2023) describe any ToT system by four choices. The class lab answered them for Game of 24. MosAIc answers them like this:

| Question | Game of 24 (lab) | MosAIc |
| --- | --- | --- |
| What is one thought? | one equation | one reading of one evidence item: effect (fall, gain, wider spread, no effect), size and a one-sentence reason |
| Generator: sample or propose? | propose k next steps in one prompt | **propose**: the model gives two readings of an item in one call, and code always adds "no effect" as the third |
| Evaluator: value or vote? | value: sure / likely / impossible | **value**: each partial ledger gets a score from base rate, novelty and consistency |
| Search | ToT-BFS with breadth b = 5, depth 3 | ToT-BFS (beam search) with breadth 3, depth up to 5 |

The lab used exact Python checks (`solve24`, `parse_equation`) wherever it could and asked the model only when code could not decide. MosAIc does the same: code checks dates, novelty, base rates and the cap on the total shift, and the model is used only to propose readings and to check that a passage really says what a reading claims.

In [ ]:
!pip install -q sentence-transformers transformers accelerate beautifulsoup4 lxml yfinance scipy

## 2. Settings

Everything that can change a result is set here.

In [ ]:
# --- Required: the SEC asks every automated request to identify itself ---
USER_AGENT = "Your Name your.email@example.com"

# Stocks to forecast, and the stocks whose filing history feeds case memory
TICKERS = ["AAPL", "NKE", "INTC"]
CASE_TICKERS = ["AAPL", "MSFT", "NVDA", "INTC", "GOOGL", "META", "NFLX", "AMZN", "TSLA", "NKE",
                "JPM", "GS", "V", "JNJ", "UNH", "LLY", "PG", "KO", "WMT", "XOM", "CVX",
                "CAT", "BA", "LMT", "NEE", "DUK", "PLD", "AMT", "LIN", "FCX"]

# Forecast dates: five test scenarios in this window (section 16).
# The window ends early enough for the 21-trading-day outcome to be known.
START_DATE, END_DATE = "2025-01-01", "2026-09-25"
DAYS_AFTER_RESULTS = 2            # "after results" and "event" scenarios: calendar days after the filing
MID_QUARTER_DAYS = 30             # "mid-quarter": trading days after the results reaction
RESULTS_COMING_DAYS = 55          # "results coming": trading days after the results reaction
EVENT_ITEMS = ("1.01", "1.02", "1.05", "2.01", "2.05", "2.06", "5.02")   # deals, cyber incidents, acquisitions,
                                  # restructuring, impairments, management changes
RANDOM_PER_TICKER = 8             # "random": Mondays drawn at random
SCENARIO_SEED = 0

# Forecast definition (Checkpoint 1.1)
HORIZON = 21              # trading days
SIGMA_LOOKBACK = 252      # one year of daily returns
BUCKET_K = 0.5            # fall < -0.5 sigma, gain > +0.5 sigma

# Documents and retrieval (Checkpoint 3)
FORMS = {"10-Q", "10-K", "8-K"}
CHUNK_WORDS, OVERLAP_WORDS = 150, 30
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
FRESH_DAYS = 30
K_CANDIDATES, K_FINAL, MMR_LAMBDA = 20, 5, 0.7
MIN_SCORE = 0.20          # searches: below this best score, "nothing relevant found" (0.30 in Checkpoint 3
                          # discarded most real evidence in the first scenario run; to tune on labelled examples)
FILING_MIN_SCORE = 0.10   # every recent filing is a candidate; only near-empty passages fall below this
QUERIES = [                       # fixed searches, in place of the ReAct loop
    "quarterly results revenue and guidance outlook",
    "legal proceedings, investigations and major risks",
    "acquisitions, agreements, restructuring and management changes",
]

# Ledger (Checkpoint 1.1 and 4)
STRENGTH_POINTS = {"small": 3, "medium": 6, "large": 9}
MAX_TOTAL_SHIFT = 15      # points per outcome
MIN_PROB = 1.0            # percent; every outcome stays possible

# Tree of Thought (Checkpoint 4)
BEAM_WIDTH = 3
PRUNE_MARGIN = 0.3
MAX_DEPTH = 5
MAX_MODEL_CALLS = 25
KEEP_NO_EFFECT_NODE = True        # the all-"no effect" ledger (the baseline) is always kept
WEIGHTS = {"base_rate": 0.4, "novelty": 0.4, "consistency": 0.2}   # defaults; tuning needs option data
SAME_STORY_SIMILARITY = 0.85      # two passages this similar describe the same story
NOVELTY_SIM_RANGE = (0.5, 0.95)   # similarity to the previous filing: <= 0.5 fully new, >= 0.95 fully repeated
                                  # (a starting guess: check the diagnostic in section 9 on the first run)

# Case memory
PRIOR_STRENGTH = 20       # pseudo-events pulling small samples toward the overall frequencies
BASE_RATE_SCALE = 0.02    # how fast the base-rate score moves away from 0.5

# Language model: "hf" (local Hugging Face model), "openai" or "anthropic"
LLM_BACKEND = "hf"
LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"     # or "Qwen/Qwen2.5-7B-Instruct" for better readings

BRANCH_LOG = "branch_log.jsonl"
TRACE_FILE = "traces.jsonl"

In [ ]:
import json, math, re, time
from dataclasses import dataclass, field
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
from scipy.stats import norm

OUTCOMES = ("fall", "flat", "gain")
EFFECTS = ("fall", "gain", "spread", "none")

## 3. Filings from SEC EDGAR

Reused from Checkpoint 3, with two changes:

- **Full filing history.** EDGAR lists only about a year of recent filings in one file and the rest in extra pages. Case memory needs years of 8-Ks, so all pages are read. Only the list of filings is needed for case memory, not the documents, so this is quick even for 30 stocks.
- **Download by date.** The documents for the forecast stocks are downloaded for the whole test window, plus 200 days before it, so each filing's previous filing is available for the novelty check.

In [ ]:
import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

HEADERS = {"User-Agent": USER_AGENT, "Accept-Encoding": "gzip, deflate"}

def sec_get(url):
    time.sleep(0.15)  # well under the SEC's limit of 10 requests per second
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r

def ticker_to_cik():
    data = sec_get("https://www.sec.gov/files/company_tickers.json").json()
    return {v["ticker"].upper(): int(v["cik_str"]) for v in data.values()}

KEYS = ["form", "filingDate", "acceptanceDateTime", "accessionNumber", "primaryDocument", "items"]

def _rows(block):
    n = len(block["form"])
    return [{k: block.get(k, [""] * n)[i] for k in KEYS} for i in range(n)]

def list_filings(cik):
    """All filings of one company, including the older pages EDGAR keeps in separate files."""
    sub = sec_get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json").json()
    rows = _rows(sub["filings"]["recent"])
    for page in sub["filings"].get("files", []):
        rows += _rows(sec_get(f"https://data.sec.gov/submissions/{page['name']}").json())
    return sub.get("name", ""), rows

def filing_base(cik, accession):
    return f"https://www.sec.gov/Archives/edgar/data/{cik}/{accession.replace('-', '')}"

def exhibit_urls(cik, accession):
    idx = sec_get(filing_base(cik, accession) + "/index.json").json()
    names = [it["name"] for it in idx["directory"]["item"]]
    return [filing_base(cik, accession) + "/" + n for n in names
            if re.search(r"ex-?99", n, re.I) and n.lower().endswith((".htm", ".html", ".txt"))]

In [ ]:
def parse_accepted(s):
    # e.g. "2024-08-01T16:30:42.000Z" -> timezone-aware UTC datetime
    return datetime.fromisoformat(s.replace("Z", "+00:00")).astimezone(timezone.utc)

def html_to_text(html):
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style"]):
        tag.decompose()
    for tag in soup.find_all(lambda t: t.name and t.name.lower() == "ix:header"):
        tag.decompose()  # hidden XBRL data in modern filings
    text = soup.get_text("\n")
    text = re.sub(r"[ \t\xa0]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()

ITEM_RE = re.compile(r"\n\s*(item\s+\d+[a-z]?\.?)\s*([^\n]{0,80})", re.I)

def split_sections(text, label):
    if label == "Press release":
        return [("Press release", text)]
    text = "\n" + text
    matches = list(ITEM_RE.finditer(text))
    if not matches:
        return [("Full text", text.strip())]
    out = []
    for j, m in enumerate(matches):
        end = matches[j + 1].start() if j + 1 < len(matches) else len(text)
        body = text[m.start():end]
        if len(body.split()) > 40:              # skips table-of-contents entries
            title = re.sub(r"\s+", " ", (m.group(1) + " " + m.group(2)).strip())[:80]
            out.append((title, body))
    return out or [("Full text", text.strip())]

def section_key(section):
    """'Item 1A. Risk Factors' -> 'risk factors', so the same section can be found in the previous filing.

    The title is used instead of the item number, because 10-Qs and 10-Ks number their items
    differently (legal proceedings are Item 1 of part II in a 10-Q, Item 3 in a 10-K).
    """
    title = re.sub(r"^\s*item\s+\d+[a-z]?\.?", "", section.strip(), flags=re.I)
    words = re.findall(r"[a-z]+", title.lower())
    if words:
        return " ".join(words[:4])
    m = re.match(r"\s*item\s+(\d+[a-z]?)", section, re.I)
    return f"item {m.group(1).lower()}" if m else section.strip().lower()

def chunk_words(text, size=None, overlap=None):
    size = size or CHUNK_WORDS
    overlap = OVERLAP_WORDS if overlap is None else overlap
    words, step, out = text.split(), size - overlap, []
    for i in range(0, max(len(words) - overlap, 1), step):
        piece = words[i:i + size]
        if len(piece) >= 30:
            out.append(" ".join(piece))
    return out

def event_key(form, items):
    """Event type used by case memory: the 8-K item number, or the report type."""
    if form != "8-K":
        return form
    codes = [c.strip() for c in str(items).split(",") if c.strip()]
    codes = [c for c in codes if c not in ("9.01",)]          # 9.01 = exhibits only, never the news
    return "8-K " + (codes[0] if codes else "other")

In [ ]:
CIK = ticker_to_cik()
FILINGS, NAMES = {}, {}
for t in sorted(set(CASE_TICKERS) | set(TICKERS)):
    NAMES[t], rows = list_filings(CIK[t])
    FILINGS[t] = [r for r in rows if r["form"] in FORMS]
    print(f"{t}: {len(FILINGS[t])} filings back to {min(r['filingDate'] for r in FILINGS[t])}")

def build_corpus(tickers, since, until):
    corpus = []
    for t in tickers:
        cik = CIK[t]
        for r in FILINGS[t]:
            accepted = parse_accepted(r["acceptanceDateTime"])
            if not (since <= accepted < until):
                continue
            docs = [(filing_base(cik, r["accessionNumber"]) + "/" + r["primaryDocument"], "Main document")]
            if r["form"] == "8-K":
                docs += [(u, "Press release") for u in exhibit_urls(cik, r["accessionNumber"])]
            for url, label in docs:
                try:
                    text = html_to_text(sec_get(url).text)
                except Exception as e:
                    print("  skipped", url, e)
                    continue
                for section, body in split_sections(text, label):
                    for piece in chunk_words(body):
                        corpus.append({
                            "id": f"{t}-{r['form']}-{r['filingDate']}-{len(corpus)}",
                            "ticker": t, "company": NAMES[t], "form": r["form"], "section": section,
                            "section_key": section_key(section), "event": event_key(r["form"], r.get("items", "")),
                            "accepted": accepted.isoformat(), "filing": r["accessionNumber"], "url": url, "text": piece,
                            "header": f"{NAMES[t]} | {r['form']} | {section} | {accepted.date()}",
                        })
        print(f"{t}: {sum(c['ticker'] == t for c in corpus)} chunks")
    return corpus

WINDOW_START = datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc) - timedelta(days=200)
WINDOW_END = datetime.fromisoformat(END_DATE).replace(tzinfo=timezone.utc) + timedelta(days=1)
corpus = build_corpus(TICKERS, WINDOW_START, WINDOW_END)
json.dump(corpus, open("corpus.json", "w"))
print("Total chunks:", len(corpus))

## 4. Index and search

Same as Checkpoint 3: each chunk is embedded as `header [SEP] text`, search keeps only chunks for the stock from the 30 days before the forecast date, takes the 20 closest and reranks them to 5 with MMR (λ = 0.7). The MMR function now takes the relevance scores directly, so the same code can be reused for evidence selection below.

In [ ]:
@dataclass
class Index:
    chunks: list
    emb: np.ndarray                 # one normalised row per chunk, so a dot product is cosine similarity
    accepted: list = field(default_factory=list)
    tickers: np.ndarray = None

    def __post_init__(self):
        self.accepted = [datetime.fromisoformat(c["accepted"]) for c in self.chunks]
        self.tickers = np.array([c["ticker"] for c in self.chunks])

def mmr_select(rel, cand_emb, k, lam):
    """Maximal Marginal Relevance: pick k items that are relevant (rel) but not near-copies."""
    rel = np.asarray(rel, dtype=float)
    selected, remaining = [], list(range(len(rel)))
    while remaining and len(selected) < k:
        if not selected:
            best = max(remaining, key=lambda i: rel[i])
        else:
            sims = cand_emb[remaining] @ cand_emb[selected].T
            scores = lam * rel[remaining] - (1 - lam) * sims.max(axis=1)
            best = remaining[int(np.argmax(scores))]
        selected.append(best)
        remaining.remove(best)
    return selected

def search_index(index, qvec, ticker, before, since=None):
    rows = np.array([i for i in range(len(index.chunks))
                     if index.tickers[i] == ticker and index.accepted[i] < before
                     and (since is None or index.accepted[i] >= since)], dtype=int)
    if len(rows) == 0:
        return {"status": "no documents in this date range", "results": []}
    sims = index.emb[rows] @ qvec
    order = rows[np.argsort(-sims)[:K_CANDIDATES]]
    best = float(index.emb[order[0]] @ qvec)
    if best < MIN_SCORE:
        return {"status": f"nothing relevant found (best score {best:.2f})", "results": []}
    rel = index.emb[order] @ qvec
    picked = mmr_select(rel, index.emb[order], K_FINAL, MMR_LAMBDA)
    return {"status": "ok", "results": [{**index.chunks[order[j]], "row": int(order[j]),
                                         "score": round(float(rel[j]), 3)} for j in picked]}

In [ ]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer(EMBED_MODEL)
emb = embedder.encode([c["header"] + " [SEP] " + c["text"] for c in corpus],
                      batch_size=64, normalize_embeddings=True, show_progress_bar=True)
INDEX = Index(corpus, np.asarray(emb, dtype=np.float32))
print("Index:", INDEX.emb.shape)

def embed_query(text):
    return embedder.encode([text], normalize_embeddings=True)[0]

## 5. Choosing the five evidence items (new)

The research step runs three searches, each returning up to 5 passages, but ToT reads at most 5 items, one tree level each. The design decision (Checkpoint 4, section 2):

1. **Pool** the passages from all searches. A passage found by several searches keeps its best score.
2. **One passage per filing**, the best one, since the critic counts each filing once anyway.
3. **MMR to 5** on what is left, with relevance = the best score, so the five do not repeat each other.

The five are then ordered newest first, which is the order the tree reads them in.

**Every recent filing is a candidate.** The scenario run showed the searches alone lose most of the evidence: in 81 of 114 forecasts the company's filings from the last 30 days were there, but all three searches scored them just under the cut-off, so nothing was kept. Event 8-Ks are short and use formal titles ("Departure of Directors or Certain Officers") that the fixed searches do not match. Within 30 days one stock usually has only one to four filings, so each one now enters the pool directly: from each filing, the passage closest in meaning to the filing's own item title (or, for a 10-Q or 10-K, to the main topics of a report), skipping signature pages and form boilerplate. The three searches stay as an extra source for long reports.

In [ ]:
ITEM_TITLES = {
    "1.01": "Entry into a Material Definitive Agreement",
    "1.02": "Termination of a Material Definitive Agreement",
    "1.05": "Material Cybersecurity Incidents",
    "2.01": "Completion of Acquisition or Disposition of Assets",
    "2.02": "Results of Operations and Financial Condition, quarterly results, revenue and outlook",
    "2.03": "Creation of a Direct Financial Obligation",
    "2.05": "Costs Associated with Exit or Disposal Activities, restructuring",
    "2.06": "Material Impairments",
    "5.02": "Departure of Directors or Certain Officers; Election of Directors; Appointment of Certain Officers",
    "5.07": "Submission of Matters to a Vote of Security Holders",
    "7.01": "Regulation FD Disclosure",
    "8.01": "Other Events",
}
REPORT_TOPICS = "results of operations, outlook and guidance, risk factors, legal proceedings"

BOILERPLATE = re.compile(
    r"pursuant to the requirements of the securities exchange act|check the appropriate box|"
    r"securities registered pursuant to section 12|cover page interactive data|inline xbrl|"
    r"indicate by check mark|exhibit index|forward-looking statements", re.I)

def is_boilerplate(text):
    return bool(BOILERPLATE.search(text))

def reference_text(chunk):
    if chunk["form"] in ("10-Q", "10-K"):
        return REPORT_TOPICS
    code = chunk["event"].replace("8-K ", "")
    return ITEM_TITLES.get(code, "company announcement")

def filing_candidates(index, ticker, as_of, since, embed_fn):
    """One passage per filing in the window: the one closest to the filing's own subject."""
    by_filing = {}
    for i, c in enumerate(index.chunks):
        if c["ticker"] == ticker and since <= index.accepted[i] < as_of and not is_boilerplate(c["text"]):
            by_filing.setdefault(c["filing"], []).append(i)
    out, refs = [], {}
    for filing, rows in by_filing.items():
        ref = reference_text(index.chunks[rows[0]])
        if ref not in refs:
            refs[ref] = embed_fn(ref)
        sims = index.emb[rows] @ refs[ref]
        j = int(np.argmax(sims))
        if sims[j] >= FILING_MIN_SCORE:
            out.append({**index.chunks[rows[j]], "row": int(rows[j]), "score": round(float(sims[j]), 3)})
    return {"status": "ok" if out else "no filings in this date range", "results": out}

def select_evidence(index, results_by_query, k=None, lam=None):
    k = k or min(K_FINAL, MAX_DEPTH)
    lam = MMR_LAMBDA if lam is None else lam
    pooled = {}
    for res in results_by_query:
        for r in res["results"]:
            if r["id"] not in pooled or r["score"] > pooled[r["id"]]["score"]:
                pooled[r["id"]] = r
    per_filing = {}
    for r in pooled.values():
        key = (r["ticker"], r["filing"])
        if key not in per_filing or r["score"] > per_filing[key]["score"]:
            per_filing[key] = r
    cands = sorted(per_filing.values(), key=lambda r: -r["score"])
    if not cands:
        return []
    rows = [r["row"] for r in cands]
    picked = mmr_select([r["score"] for r in cands], index.emb[rows], k, lam)
    chosen = [cands[j] for j in picked]
    return sorted(chosen, key=lambda r: (r["accepted"], r["score"]), reverse=True)

## 6. Prices, buckets and the baseline

Reused from Checkpoint 3. The forecast date means the start of that day, so only closes on earlier days are known. A "large" move is more than half of the stock's typical monthly move over the past year (σ, one year of daily log returns scaled to 21 days). The baseline uses recent volatility, plus the typical earnings-day jump when the next results are likely within the month.

In [ ]:
def prices_before(px, as_of):
    return px[np.array([d < as_of.date() for d in px.index.date])].dropna()

def thresholds(px, as_of):
    past = prices_before(px, as_of)
    r = np.diff(np.log(past.to_numpy(dtype=float)))[-SIGMA_LOOKBACK:]
    if len(r) < SIGMA_LOOKBACK // 2:
        raise ValueError(f"not enough price history before {as_of.date()}")
    sigma = float(r.std() * math.sqrt(HORIZON))
    return {"sigma": sigma, "lower": -BUCKET_K * sigma, "upper": BUCKET_K * sigma, "start_price": float(past.iloc[-1])}

def realised_return(px, as_of):
    past = prices_before(px, as_of)
    after = px[np.array([d >= as_of.date() for d in px.index.date])].dropna()
    if past.empty or len(after) < HORIZON:
        return None
    return float(math.log(after.iloc[HORIZON - 1] / past.iloc[-1]))

def bucket_of(ret, th):
    return "fall" if ret < th["lower"] else "gain" if ret > th["upper"] else "flat"

def realised_bucket(px, as_of):
    ret = realised_return(px, as_of)
    return None if ret is None else bucket_of(ret, thresholds(px, as_of))

def typical_earnings_move(px, dates):
    moves = []
    for d in dates[-4:]:
        before_px = px[px.index.date <= d.date()]
        after_px = px[px.index.date > d.date()]
        if len(before_px) and len(after_px):
            moves.append(abs(math.log(after_px.iloc[0] / before_px.iloc[-1])))
    return float(np.mean(moves)) if moves else None

def baseline_probabilities(px, as_of, results_dates):
    """Fall / flat / gain in percent, from prices and the earnings calendar only."""
    info = {}
    try:
        past = prices_before(px, as_of)
        r = np.diff(np.log(past.to_numpy(dtype=float)))[-SIGMA_LOOKBACK:]
        th = thresholds(px, as_of)
        w = 0.94 ** np.arange(len(r))[::-1]                       # recent days weigh more
        var_month = float(np.sum(w * r ** 2) / np.sum(w)) * HORIZON
        past_results = sorted(d for d in results_dates if d < as_of)
        if past_results and as_of < past_results[-1] + timedelta(days=91) <= as_of + timedelta(days=30):
            move = typical_earnings_move(px, past_results)
            if move:
                var_month += move ** 2
                info["earnings_move"] = round(move, 4)
        info["earnings_in_window"] = "earnings_move" in info
        s = math.sqrt(var_month)
        fall, gain = norm.cdf(th["lower"] / s), 1 - norm.cdf(th["upper"] / s)
        probs = {"fall": fall, "flat": 1 - fall - gain, "gain": gain}
        info.update({"sigma_year": round(th["sigma"], 4), "sigma_month": round(s, 4)})
    except Exception as e:
        probs = {"fall": 0.3085, "flat": 0.383, "gain": 0.3085}
        info["fallback"] = str(e)
    return {k: round(100 * float(v), 2) for k, v in probs.items()}, info

def main_results(r202, reports, max_gap_days=75):
    """Keep one results announcement per quarter.

    Some companies also use item 2.02 for mid-quarter updates (Chevron's "interim update", for example).
    The real results come just before the quarter's 10-Q or 10-K, so for each report the last item-2.02
    filing in the 75 days before it is kept. Results announced after the latest report are kept too.
    """
    r202 = sorted(set(r202))
    kept = set()
    for rep in reports:
        before = [a for a in r202 if rep - timedelta(days=max_gap_days) <= a <= rep]
        if before:
            kept.add(before[-1])
    last_rep = max(reports) if reports else None
    tail = [a for a in r202 if last_rep is None or a > last_rep]
    if tail:
        kept.add(tail[-1])
    return sorted(kept)

def results_dates(filings):
    """Acceptance times of the quarterly results 8-Ks (item 2.02), one per quarter."""
    r202 = [parse_accepted(r["acceptanceDateTime"]) for r in filings
            if r["form"] == "8-K" and "2.02" in str(r.get("items", ""))]
    reports = [parse_accepted(r["acceptanceDateTime"]) for r in filings if r["form"] in ("10-Q", "10-K")]
    return main_results(r202, reports)

In [ ]:
import yfinance as yf

PRICES = {}
for t in sorted(set(CASE_TICKERS) | set(TICKERS)):
    px = yf.download(t, start="2012-01-01", progress=False, auto_adjust=True)["Close"]
    PRICES[t] = px.squeeze().dropna()
    print(f"{t}: {len(PRICES[t])} days from {PRICES[t].index[0].date()}")

## 7. Ledger arithmetic

Each reading becomes a line that shifts the three probabilities by small (3), medium (6) or large (9) points, and every line sums to zero:

| Effect | Fall | Flat | Gain |
| --- | --- | --- | --- |
| fall, size s | +s | −s/3 | −2s/3 |
| gain, size s | −2s/3 | −s/3 | +s |
| wider spread (new), size s | +s/2 | −s | +s/2 |
| no effect | 0 | 0 | 0 |

If the lines together move any outcome by more than 15 points, all lines are scaled down together. Every outcome keeps at least 1%.

In [ ]:
def shift_for(effect, size):
    if effect == "none":
        return {"fall": 0.0, "flat": 0.0, "gain": 0.0}
    s = float(STRENGTH_POINTS[size])
    return {"fall": {"fall": s, "flat": -s / 3, "gain": -2 * s / 3},
            "gain": {"fall": -2 * s / 3, "flat": -s / 3, "gain": s},
            "spread": {"fall": s / 2, "flat": -s, "gain": s / 2}}[effect]

def apply_ledger(base, readings):
    """Baseline plus the readings' shifts, capped and floored. Returns (final, cap_factor)."""
    total = {k: sum(shift_for(r["effect"], r["size"])[k] for r in readings) for k in OUTCOMES}
    biggest = max(abs(v) for v in total.values())
    factor = MAX_TOTAL_SHIFT / biggest if biggest > MAX_TOTAL_SHIFT else 1.0
    raw = {k: max(base[k] + factor * total[k], 0.0) for k in OUTCOMES}
    return floor_and_normalise(raw), factor

def floor_and_normalise(p):
    """Scale to 100 while keeping every outcome at MIN_PROB or more (floors hold after rescaling)."""
    floored = set()
    while True:
        free = [k for k in OUTCOMES if k not in floored]
        room = 100.0 - MIN_PROB * len(floored)
        s = sum(p[k] for k in free)
        out = {k: (MIN_PROB if k in floored else (room * p[k] / s if s > 0 else room / len(free))) for k in OUTCOMES}
        low = [k for k in free if out[k] < MIN_PROB]
        if not low:
            return {k: round(v, 2) for k, v in out.items()}
        floored.update(low)

def total_shift(readings):
    """Size of a ledger's move away from the baseline, used to break ties (smaller wins)."""
    total = {k: sum(shift_for(r["effect"], r["size"])[k] for r in readings) for k in OUTCOMES}
    return sum(abs(v) for v in total.values())

## 8. Case memory: base rates

**The question behind the score.** For a reading like "this management change points to a fall, medium size", case memory asks: *if we had applied this shift after every past event of the same type, would those forecasts have scored better?* This checks both the direction and the size of a reading against history, which is what the Checkpoint 3 feedback asked for.

**How it works.**

1. **Events** are past filings of the 30 stocks, labelled by type: the 8-K item number (2.02 results, 5.02 management change, 1.01 agreement, and so on) or the report type (10-Q, 10-K).
2. **Same timing as the forecast.** If the evidence was filed 9 days before the forecast date, each past event's 21-day window also starts 9 days after that event. So results already reflected in the price count as already priced in.
3. **No look-ahead.** Only past events whose 21-day window ended before the forecast date are used.
4. **Small samples are pulled toward the overall frequencies** (the share of all 21-day windows that ended in a fall, flat month or gain), with a weight of 20 pseudo-events. A rare event type therefore says little, and "no effect" wins.
5. **The score** is the average improvement in log score from applying the shift, mapped to 0–1. "No effect" always scores 0.5, the neutral point.

In [ ]:
class CaseMemory:
    def __init__(self, events, prices):
        """events: {ticker: [(event_key, accepted_datetime), ...]}; prices: {ticker: Series of closes}."""
        self.events, self.cache = events, {}
        self.px = {}
        for t, s in prices.items():
            s = s.dropna()
            self.px[t] = (np.array(s.index.values, dtype="datetime64[D]"), np.log(s.to_numpy(dtype=float)))

    def window(self, ticker, start_date):
        """Bucket of the 21 trading days from start_date, and the date the window ends."""
        key = (ticker, start_date)
        if key not in self.cache:
            dates, logp = self.px[ticker]
            i = int(np.searchsorted(dates, np.datetime64(start_date, "D")))   # first close on or after start
            if i < SIGMA_LOOKBACK // 2 + 1 or i + HORIZON - 1 >= len(dates):
                self.cache[key] = (None, None)
            else:
                r = np.diff(logp[max(0, i - 1 - SIGMA_LOOKBACK):i])
                sigma = float(r.std() * math.sqrt(HORIZON))
                ret = float(logp[i + HORIZON - 1] - logp[i - 1])
                b = "fall" if ret < -BUCKET_K * sigma else "gain" if ret > BUCKET_K * sigma else "flat"
                self.cache[key] = (b, pd.Timestamp(dates[i + HORIZON - 1]).date())
        return self.cache[key]

    def overall(self, as_of):
        """Share of all 21-day windows (weekly starts, all stocks) ending in each bucket, before as_of."""
        counts = dict.fromkeys(OUTCOMES, 0)
        for t, (dates, _) in self.px.items():
            for d in dates[SIGMA_LOOKBACK::5]:
                b, end = self.window(t, pd.Timestamp(d).date())
                if b and end < as_of.date():
                    counts[b] += 1
        n = sum(counts.values())
        return {k: counts[k] / n for k in OUTCOMES} if n else {"fall": 0.3085, "flat": 0.383, "gain": 0.3085}

    def distribution(self, event, lag_days, as_of, p0=None):
        """Smoothed bucket frequencies after past events of this type, and the number of events used."""
        p0 = p0 or self.overall(as_of)
        counts = dict.fromkeys(OUTCOMES, 0)
        for t, evs in self.events.items():
            for key, accepted in evs:
                if key != event or accepted >= as_of:
                    continue
                b, end = self.window(t, accepted.date() + timedelta(days=lag_days))
                if b and end < as_of.date():
                    counts[b] += 1
        n = sum(counts.values())
        q = {k: (counts[k] + PRIOR_STRENGTH * p0[k]) / (n + PRIOR_STRENGTH) for k in OUTCOMES}
        return q, n, p0

    def score(self, effect, size, event, lag_days, as_of, p0=None):
        """0-1 score; 0.5 = no evidence either way. Returns (score, details)."""
        if effect == "none":
            return 0.5, {"n_events": None}
        q, n, p0 = self.distribution(event, lag_days, as_of, p0)
        s = shift_for(effect, size)
        p1 = {k: max(p0[k] + s[k] / 100, 0.01) for k in OUTCOMES}
        z = sum(p1.values())
        gain = sum(q[k] * (math.log(p1[k] / z) - math.log(p0[k])) for k in OUTCOMES)
        return 0.5 + 0.5 * math.tanh(gain / BASE_RATE_SCALE), {"n_events": n, "log_gain": gain, "q": q}

## 9. Novelty

Companies repeat the same wording every quarter. A passage is compared with the same section of the company's previous filing of the same kind (the previous 10-Q or 10-K for a report section, the previous 8-K with the same item for an 8-K), and its highest similarity to any chunk there is turned into a novelty score from 0 to 1. When there is no previous filing in the index, the passage counts as new and this is flagged.

**Why the similarity is rescaled.** Passages from the same company and section are always fairly similar in embedding space, even when the content is new. Using 1 minus similarity directly would make almost everything look repeated, and "no effect" would win nearly every time. So similarity of 0.5 or less counts as fully new, 0.95 or more as fully repeated, with a straight line in between. These two cut-offs are a starting guess; the diagnostic cell after the run shows the real values.

For the readings: a reading that claims an effect gets the novelty score, and "no effect" gets 1 minus it. Repeated wording therefore favours "no effect".

In [ ]:
def previous_filing_rows(index, item):
    family = ("10-Q", "10-K") if item["form"] in ("10-Q", "10-K") else ("8-K",)
    acc = datetime.fromisoformat(item["accepted"])
    rows = [i for i, c in enumerate(index.chunks)
            if c["ticker"] == item["ticker"] and c["form"] in family and c["filing"] != item["filing"]
            and c["section_key"] == item["section_key"] and index.accepted[i] < acc
            and (c["form"] != "8-K" or c["event"] == item["event"])]
    if not rows:
        return [], None
    latest = max(index.accepted[i] for i in rows)
    prev_filing = next(index.chunks[i]["filing"] for i in rows if index.accepted[i] == latest)
    return [i for i in rows if index.chunks[i]["filing"] == prev_filing], prev_filing

def novelty(index, item):
    """(novelty 0-1, previous filing id or None)."""
    rows, prev = previous_filing_rows(index, item)
    if not rows:
        return 1.0, None
    sim = float(np.max(index.emb[rows] @ index.emb[item["row"]]))
    lo, hi = NOVELTY_SIM_RANGE
    return float(min(max((hi - sim) / (hi - lo), 0.0), 1.0)), prev

def reading_novelty(effect, nov):
    return 1.0 - nov if effect == "none" else nov

In [ ]:
EVENTS = {t: [(event_key(r["form"], r.get("items", "")), parse_accepted(r["acceptanceDateTime"])) for r in FILINGS[t]]
          for t in CASE_TICKERS}
CASE = CaseMemory(EVENTS, {t: PRICES[t] for t in CASE_TICKERS})
print("Event types with the most history:")
print(pd.Series([k for evs in EVENTS.values() for k, _ in evs]).value_counts().head(12))

## 10. The language model

One small wrapper, so the backend can be switched in the settings: a local Hugging Face model (Qwen2.5-3B by default, free on a Colab T4), or an API model. Decoding is greedy (temperature 0), so the same input always gives the same output. The lab sampled at temperature 0.7; here reproducible runs matter more.

The class lab found that the generator model is the bottleneck: the paper's ToT solved 74% of Game of 24 puzzles with GPT-4 and 19% with GPT-3.5. Expect weaker readings from a 3B model than from a 7B or API model.

In [ ]:
class Model:
    """Counts every call, so the 25-call budget per forecast can be checked."""

    def __init__(self, chat_fn):
        self.chat_fn, self.calls = chat_fn, 0

    def __call__(self, messages, max_tokens=300):
        self.calls += 1
        return self.chat_fn(messages, max_tokens)

def parse_json(reply, kind="["):
    """First JSON list ("[") or object ("{") in a reply, tolerating code fences and extra text."""
    reply = re.sub(r"```(?:json)?", "", reply)
    pattern = r"\[.*\]" if kind == "[" else r"\{.*\}"
    m = re.search(pattern, reply, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None

In [ ]:
def make_chat(backend, model_name):
    if backend == "hf":
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
        tok = AutoTokenizer.from_pretrained(model_name)
        lm = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="auto")

        def chat(messages, max_tokens):
            prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            inputs = tok(prompt, return_tensors="pt").to(lm.device)
            out = lm.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
            return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        return chat

    if backend == "openai":                      # needs OPENAI_API_KEY in the environment
        from openai import OpenAI
        client = OpenAI()

        def chat(messages, max_tokens):
            r = client.chat.completions.create(model=model_name, messages=messages,
                                               max_tokens=max_tokens, temperature=0)
            return r.choices[0].message.content.strip()
        return chat

    if backend == "anthropic":                   # needs ANTHROPIC_API_KEY in the environment
        import anthropic
        client = anthropic.Anthropic()

        def chat(messages, max_tokens):
            system = "\n".join(m["content"] for m in messages if m["role"] == "system")
            rest = [m for m in messages if m["role"] != "system"]
            r = client.messages.create(model=model_name, system=system, messages=rest,
                                       max_tokens=max_tokens, temperature=0)
            return r.content[0].text.strip()
        return chat

    raise ValueError(f"unknown backend {backend!r}")

CHAT = make_chat(LLM_BACKEND, LLM_MODEL)
print("Model ready:", LLM_BACKEND, LLM_MODEL)

## 11. Thought generator: proposing readings

The **propose** strategy from the paper: one call per evidence item returns the two most plausible readings, most plausible first. Code then adds "no effect" if the model did not include it, so every item has a neutral reading and the all-"no effect" ledger (the baseline) always exists.

The model only proposes the effect, the size and a reason. It never writes numbers: the shifts come from the table in section 7.

In [ ]:
GEN_SYSTEM = """You help forecast a stock's move over the next month (21 trading days).
You are shown ONE passage from the company's SEC filings, published before the forecast date.
Give the TWO most plausible readings of how this passage affects the stock over the next month, most plausible first.
Each reading has:
- "effect": "fall" (points to a large fall), "gain" (points to a large gain),
  "spread" (makes a large move in EITHER direction more likely, e.g. an uncertain lawsuit or a big product launch),
  or "none" (routine wording, already known, or already in the price)
- "size": "small", "medium" or "large"
- "reason": one sentence that quotes or closely paraphrases the passage
The two readings must have different effects.
Reply with ONLY a JSON list of two objects."""

NO_EFFECT_REASON = "Routine or already reflected in the price; no change to the outlook."

def item_message(item, as_of):
    days = (as_of.date() - datetime.fromisoformat(item["accepted"]).date()).days
    return (f"Company: {item['company']} ({item['ticker']}). Forecast date: {as_of.date()}.\n"
            f"{item['form']} | {item['section']} | filed {item['accepted'][:10]} ({days} days before the forecast date)\n\n"
            f"{item['text']}")

def propose_readings(model, item, as_of):
    """Up to 3 readings: the model's (most plausible first), plus 'no effect' if missing."""
    reply = model([{"role": "system", "content": GEN_SYSTEM},
                   {"role": "user", "content": item_message(item, as_of)}], max_tokens=300)
    data = parse_json(reply, "[") or []
    readings, seen = [], set()
    for d in data if isinstance(data, list) else []:
        if not isinstance(d, dict):
            continue
        effect = str(d.get("effect", "")).lower().strip()
        size = str(d.get("size", "")).lower().strip()
        if effect not in EFFECTS or effect in seen or (effect != "none" and size not in STRENGTH_POINTS):
            continue
        seen.add(effect)
        readings.append({"effect": effect, "size": size if effect != "none" else "small",
                         "reason": str(d.get("reason", ""))[:300], "source": item["id"]})
        if len(readings) == 2:
            break
    if "none" not in seen:
        readings.append({"effect": "none", "size": "small", "reason": NO_EFFECT_REASON, "source": item["id"]})
    return readings, reply

## 12. Grounding check

The first scoring check is pass/fail: does the passage actually say what the reading claims? This is the only check that needs the model, and it is skipped for "no effect", which claims nothing. A reading that fails is pruned at once.

In [ ]:
GROUND_SYSTEM = """You check whether a claim is supported by a passage from an SEC filing.
Answer "yes" only if the passage states or directly implies the claim. Otherwise answer "no".
Reply with one word: yes or no."""

def grounded(model, item, reading):
    if reading["effect"] == "none":
        return True, None
    reply = model([{"role": "system", "content": GROUND_SYSTEM},
                   {"role": "user", "content": f"Passage:\n{item['text']}\n\nClaim: {reading['reason']}"}], max_tokens=5)
    word = re.findall(r"[a-z]+", reply.lower())
    return (bool(word) and word[0] == "yes"), reply

def date_ok(item, as_of):
    acc = datetime.fromisoformat(item["accepted"])
    return as_of - timedelta(days=FRESH_DAYS) <= acc < as_of

## 13. Scoring the readings of each item

All model calls happen here, once per forecast. For each of the (at most 5) evidence items, newest first:

1. **Generate:** one call proposes the readings (section 11).
2. **Hard checks:** a reading is pruned at once if its passage breaks the date rule (filed before the forecast date, within 30 days) or fails grounding (section 12).
3. **Score from data:** each surviving reading gets a base-rate score (section 8) and a novelty score (section 9). Its reading score is their weighted average.

Everything is cached, so the beam search, the comparisons and the settings sweep below run in plain code with no further model calls. A forecast uses at most 5 generation calls + 10 grounding calls = 15, inside the budget of 25. If the budget runs out anyway, the remaining items are read as "no effect".

In [ ]:
def prepare_items(model, items, as_of, case, index, p0=None):
    prepared = []
    for item in items[:MAX_DEPTH]:
        item = dict(item, vec=index.emb[item["row"]])          # used by the same-story check
        lag = (as_of.date() - datetime.fromisoformat(item["accepted"]).date()).days
        nov, prev = novelty(index, item)
        entry = {"item": item, "novelty": nov, "previous_filing": prev, "lag_days": lag, "raw": None}
        if model.calls >= MAX_MODEL_CALLS:
            readings = [{"effect": "none", "size": "small", "reason": NO_EFFECT_REASON, "source": item["id"]}]
            entry["raw"] = "budget used up"
        else:
            readings, entry["raw"] = propose_readings(model, item, as_of)
        for r in readings:
            r["ok"], r["why"] = True, ""
            if r["effect"] != "none" and not date_ok(item, as_of):
                r["ok"], r["why"] = False, "date rule"
            elif r["effect"] != "none" and model.calls >= MAX_MODEL_CALLS:
                r["ok"], r["why"] = False, "budget used up"
            elif r["effect"] != "none":
                ok, _ = grounded(model, item, r)
                if not ok:
                    r["ok"], r["why"] = False, "failed grounding"
            br, detail = case.score(r["effect"], r["size"], item["event"], lag, as_of, p0)
            r["base_rate"], r["n_events"] = br, detail.get("n_events")
            r["novelty"] = reading_novelty(r["effect"], nov)
            wb, wn = WEIGHTS["base_rate"], WEIGHTS["novelty"]
            r["score"] = (wb * r["base_rate"] + wn * r["novelty"]) / (wb + wn)
        entry["readings"] = readings
        prepared.append(entry)
    return prepared

## 14. Beam search (the controller)

ToT-BFS with breadth 3, as in Checkpoint 4:

- **Node:** the baseline plus one chosen reading for each item read so far. **Depth** = number of items read.
- **Node score** = weighted average of its readings' scores (base rate and novelty, weight 0.8) and its consistency (weight 0.2). Consistency is 1 minus penalties: 0.5 for each pair of effect readings whose passages tell the same story (similarity above 0.85), and the share of the shift removed by the 15-point cap.
- **Keep** the top 3 children at each level. **Drop** any child more than 0.3 below the best.
- **Always keep** the all-"no effect" node, so the baseline can still win. This needs at least 2 slots: with a beam of 1 the rule would fill the only slot and the search could never choose anything else.
- **Ties** go to the smaller total shift.
- **Every node is logged**, pruned ones too, with its parent, readings and score.

In [ ]:
def readings_of(prepared, choice):
    return [prepared[d]["readings"][i] for d, i in enumerate(choice)]

def consistency(prepared, choice, base):
    rs = readings_of(prepared, choice)
    effects = [(d, r) for d, r in enumerate(rs) if r["effect"] != "none"]
    dups = 0
    for a in range(len(effects)):
        for b in range(a + 1, len(effects)):
            ea, eb = prepared[effects[a][0]]["item"], prepared[effects[b][0]]["item"]
            if "vec" in ea and "vec" in eb and float(np.dot(ea["vec"], eb["vec"])) > SAME_STORY_SIMILARITY:
                dups += 1
    _, factor = apply_ledger(base, rs)
    return max(0.0, 1.0 - 0.5 * dups - (1.0 - factor))

def node_score(prepared, choice, base):
    rs = readings_of(prepared, choice)
    if not rs:
        return 0.5
    reading_part = float(np.mean([r["score"] for r in rs]))
    wc = WEIGHTS["consistency"]
    return (1 - wc) * reading_part + wc * consistency(prepared, choice, base)

def _rank_key(node):
    return (-round(node["score"], 9), round(node["shift"], 9))

def beam_search(prepared, base, beam_width=None, prune_margin="default", keep_no_effect=None, log=None):
    beam_width = beam_width or BEAM_WIDTH
    prune_margin = PRUNE_MARGIN if prune_margin == "default" else prune_margin
    keep_no_effect = KEEP_NO_EFFECT_NODE if keep_no_effect is None else keep_no_effect
    keep_no_effect = keep_no_effect and beam_width >= 2   # with one slot, the rule would always force the baseline
    root = {"id": 0, "parent": None, "depth": 0, "choice": (), "score": 0.5, "shift": 0.0, "status": "root", "why": ""}
    nodes, frontier = [root], [root]
    for depth, entry in enumerate(prepared, start=1):
        children = []
        for parent in frontier:
            for i, r in enumerate(entry["readings"]):
                choice = parent["choice"] + (i,)
                node = {"id": len(nodes), "parent": parent["id"], "depth": depth, "choice": choice,
                        "score": None, "shift": total_shift(readings_of(prepared, choice)), "status": "", "why": ""}
                nodes.append(node)
                if not r["ok"]:
                    node.update(status="pruned", why=r["why"])
                    continue
                node["score"] = node_score(prepared, choice, base)
                children.append(node)
        children.sort(key=_rank_key)
        kept = children[:beam_width]
        for n in children[beam_width:]:
            n.update(status="pruned", why=f"outside the top {beam_width}")
        if prune_margin is not None and kept:
            best = kept[0]["score"]
            for n in kept[1:]:
                if n["score"] < best - prune_margin:
                    n.update(status="pruned", why=f"more than {prune_margin} below the best")
            kept = [n for n in kept if n["status"] != "pruned"]
        if keep_no_effect:
            neutral = next((n for n in children if all(prepared[d]["readings"][i]["effect"] == "none"
                                                       for d, i in enumerate(n["choice"]))), None)
            if neutral is not None and neutral not in kept:
                if len(kept) >= beam_width:
                    kept[-1].update(status="pruned", why="replaced by the all-'no effect' node")
                    kept = kept[:-1]
                neutral.update(status="", why="")
                kept.append(neutral)
        for n in kept:
            n["status"] = "kept"
        frontier = kept or frontier
    best = sorted(frontier, key=_rank_key)[0] if prepared else root
    best["status"] = "chosen"
    ledger = [r for r in readings_of(prepared, best["choice"]) if r["effect"] != "none"]
    final, factor = apply_ledger(base, readings_of(prepared, best["choice"]))
    if log is not None:
        for n in nodes:
            log({**{k: v for k, v in n.items() if k != "choice"},
                 "readings": [(r["source"], r["effect"], r["size"]) for r in readings_of(prepared, n["choice"])]})
    return {"final": final, "ledger": ledger, "choice": best["choice"], "score": best["score"],
            "cap_factor": factor, "nodes": nodes}

## 15. The comparisons

All four use the same evidence, the same readings and the same hard checks, so differences come only from how a reading is chosen. They mirror the lab's IO / CoT / ToT benchmark:

| Variant | Lab equivalent | How the ledger is chosen |
| --- | --- | --- |
| Prompt only | IO | the model states the three probabilities from the company name and date alone (the prompt-only baseline from Checkpoint 2) |
| Single pass | CoT | for each item, the model's first reading, if it passes the hard checks (the Checkpoint 3 behaviour: commit to the first impression) |
| ToT, beam 1 | ToT b = 1 | greedy search with the scores |
| ToT, beam 3 | ToT b = 5 | the full design |

Two more, for analysis only: the **baseline** itself (every item read as "no effect"), and an **exhaustive search** that scores every possible ledger (at most 3⁵ = 243). The exhaustive search shows whether the beam misses the ledger the scoring likes best.

In [ ]:
def single_pass(prepared, base):
    choice = tuple(0 if entry["readings"][0]["ok"] else
                   next(i for i, r in enumerate(entry["readings"]) if r["effect"] == "none")
                   for entry in prepared)
    final, _ = apply_ledger(base, readings_of(prepared, choice))
    return {"final": final, "choice": choice,
            "ledger": [r for r in readings_of(prepared, choice) if r["effect"] != "none"]}

def exhaustive(prepared, base):
    """Every complete ledger made of surviving readings, best first."""
    import itertools
    options = [[i for i, r in enumerate(e["readings"]) if r["ok"]] for e in prepared]
    out = []
    for choice in itertools.product(*options):
        rs = readings_of(prepared, choice)
        out.append({"choice": choice, "score": node_score(prepared, choice, base) if rs else 0.5,
                    "shift": total_shift(rs), "final": apply_ledger(base, rs)[0]})
    return sorted(out, key=_rank_key)

PROMPT_ONLY_SYSTEM = """You forecast a stock's return over the next 21 trading days.
Give the probability (in percent) of a large fall, a roughly flat month and a large gain,
where large means more than half of the stock's typical monthly move over the past year.
Reply with ONLY a JSON object: {"fall": number, "flat": number, "gain": number}"""

def prompt_only(model, company, ticker, as_of):
    reply = model([{"role": "system", "content": PROMPT_ONLY_SYSTEM},
                   {"role": "user", "content": f"Company: {company} ({ticker}). Forecast date: {as_of.date()}."}],
                  max_tokens=60)
    d = parse_json(reply, "{") or {}
    try:
        p = {k: max(float(d[k]), 0.0) for k in OUTCOMES}
        if sum(p.values()) <= 0:
            raise ValueError
        return floor_and_normalise(p), True
    except (KeyError, TypeError, ValueError):
        return {"fall": 30.85, "flat": 38.3, "gain": 30.85}, False

## 16. Running one forecast

For one stock and date: baseline from prices, three fixed searches, evidence selection, item preparation (all model calls), then every variant on the same cached readings. The prompt-only call is counted separately, so the ToT budget covers only ToT. The outcome bucket is looked up last.

In [ ]:
def forecast_dates(filings, start, end, days_after=None):
    days_after = DAYS_AFTER_RESULTS if days_after is None else days_after
    lo = datetime.fromisoformat(start).replace(tzinfo=timezone.utc)
    hi = datetime.fromisoformat(end).replace(tzinfo=timezone.utc)
    out = []
    for d in results_dates(filings):
        as_of = datetime(d.year, d.month, d.day, tzinfo=timezone.utc) + timedelta(days=days_after)
        if lo <= as_of <= hi:
            out.append(as_of)
    return sorted(set(out))

def brier(probs, outcome):
    return sum((probs[k] / 100 - (1.0 if k == outcome else 0.0)) ** 2 for k in OUTCOMES)

def run_forecast(ticker, as_of, chat_fn, index, case, px, filings, embed_fn, company=None, branch_log=None):
    company = company or ticker
    fid = f"{ticker}-{as_of.date()}"
    t0 = time.time()
    base, info = baseline_probabilities(px, as_of, results_dates(filings))
    since = as_of - timedelta(days=FRESH_DAYS)
    searches = [search_index(index, embed_fn(q), ticker, as_of, since=since) for q in QUERIES]
    filings_in_window = filing_candidates(index, ticker, as_of, since, embed_fn)
    evidence = select_evidence(index, searches + [filings_in_window])
    in_window = sum(1 for i, c in enumerate(index.chunks)
                    if c["ticker"] == ticker and since <= index.accepted[i] < as_of)
    model = Model(chat_fn)
    p0 = case.overall(as_of)
    prepared = prepare_items(model, evidence, as_of, case, index, p0)
    t_prep = time.time() - t0

    def log_to_file(rec):
        if branch_log:
            with open(branch_log, "a") as f:
                f.write(json.dumps({"forecast": fid, **rec}, default=str) + "\n")

    t1 = time.time()
    tot3 = beam_search(prepared, base, log=log_to_file)
    t_search = time.time() - t1
    tot1 = beam_search(prepared, base, beam_width=1, keep_no_effect=False)
    single = single_pass(prepared, base)
    every = exhaustive(prepared, base)
    po_model = Model(chat_fn)
    po, po_ok = prompt_only(po_model, company, ticker, as_of)
    outcome = realised_bucket(px, as_of)
    return {
        "id": fid, "ticker": ticker, "as_of": as_of, "outcome": outcome, "baseline_info": info,
        "evidence": [{k: e[k] for k in ("id", "form", "section", "accepted", "score", "text")} for e in evidence],
        "search_check": {"chunks in window": in_window, "filings in window": len(filings_in_window["results"]),
                         **{f"best score: {q[:30]}": (max(r["score"] for r in s["results"]) if s["results"] else s["status"])
                            for q, s in zip(QUERIES, searches)}},
        "prepared": prepared, "base": base,
        "variants": {"baseline": base, "prompt only": po, "single pass": single["final"],
                     "ToT beam 1": tot1["final"], "ToT beam 3": tot3["final"]},
        "prompt_only_parsed": po_ok,
        "tot": tot3, "tot1": tot1, "single": single, "exhaustive": every,
        "calls_tot": model.calls, "seconds_prepare": round(t_prep, 1), "seconds_search": round(t_search, 3),
    }

## 17. Results

In [ ]:
def results_table(records, n_boot=2000, seed=0):
    """Mean Brier score per variant, and the paired difference with the baseline (95% interval)."""
    recs = [r for r in records if r["outcome"]]
    names = list(recs[0]["variants"])
    rng = np.random.default_rng(seed)
    rows = []
    b0 = np.array([brier(r["variants"]["baseline"], r["outcome"]) for r in recs])
    for name in names:
        b = np.array([brier(r["variants"][name], r["outcome"]) for r in recs])
        d = b - b0
        boots = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(n_boot)]
        rows.append({"variant": name, "forecasts": len(b), "mean Brier": round(float(b.mean()), 4),
                     "vs baseline": round(float(d.mean()), 4),
                     "95% interval": (round(float(np.quantile(boots, 0.025)), 4), round(float(np.quantile(boots, 0.975)), 4)),
                     "changed vs baseline": int(sum(r["variants"][name] != r["base"] for r in recs))})
    return pd.DataFrame(rows).set_index("variant")

def calls_table(records):
    return pd.DataFrame([{"forecast": r["id"], "items": len(r["prepared"]), "model calls (ToT)": r["calls_tot"],
                          "seconds (model calls)": r["seconds_prepare"], "seconds (search)": r["seconds_search"],
                          "nodes": len(r["tot"]["nodes"])} for r in records])

def sweep(records, widths=(1, 3, 5), margins=(0.1, 0.3, None)):
    """Re-run the search with other settings on the cached readings: no extra model calls."""
    rows = []
    recs = [r for r in records if r["outcome"]]
    for w in widths:
        for m in margins:
            briers, changed, nodes = [], 0, []
            for r in recs:
                res = beam_search(r["prepared"], r["base"], beam_width=w, prune_margin=m)
                briers.append(brier(res["final"], r["outcome"]))
                changed += res["choice"] != r["tot"]["choice"]
                nodes.append(len(res["nodes"]))
            rows.append({"beam width": w, "margin": "none" if m is None else m,
                         "mean Brier": round(float(np.mean(briers)), 4),
                         "ledgers different from (3, 0.3)": changed, "mean nodes": round(float(np.mean(nodes)), 1)})
    return pd.DataFrame(rows)

def search_vs_scoring(records):
    """Search error: the beam missed the ledger the scoring likes best.
    Scoring error: a ledger the scoring ranked lower turned out closer to the outcome."""
    rows = []
    for r in (r for r in records if r["outcome"]):
        every = r["exhaustive"]
        chosen = r["tot"]["choice"]
        best_by_score = every[0]["choice"]
        chosen_brier = brier(r["tot"]["final"], r["outcome"])
        best_hindsight = min(every, key=lambda e: brier(e["final"], r["outcome"]))
        scores = np.array([e["score"] for e in every])
        briers = np.array([brier(e["final"], r["outcome"]) for e in every])
        corr = float(pd.Series(scores).corr(pd.Series(-briers), method="spearman")) if len(every) > 2 else float("nan")
        rows.append({"forecast": r["id"], "ledgers possible": len(every),
                     "beam found the top-scored ledger": chosen == best_by_score,
                     "a pruned ledger was closer to the outcome": brier(best_hindsight["final"], r["outcome"]) < chosen_brier - 1e-9,
                     "score vs accuracy (rank correlation)": round(corr, 2)})
    return pd.DataFrame(rows)

def print_tree(record, max_depth=None):
    """Text view of the search tree, like the lab's visualize(): kept, chosen and pruned nodes."""
    prepared, nodes = record["prepared"], record["tot"]["nodes"]
    children = {}
    for n in nodes:
        children.setdefault(n["parent"], []).append(n)
    lines = [f"{record['id']}  baseline {record['base']}  outcome: {record['outcome']}"]

    def walk(n, indent):
        for c in children.get(n["id"], []):
            if max_depth and c["depth"] > max_depth:
                continue
            r = prepared[c["depth"] - 1]["readings"][c["choice"][-1]]
            mark = {"chosen": "★", "kept": "●"}.get(c["status"], "○")
            score = "  -  " if c["score"] is None else f"{c['score']:.2f}"
            why = f"  [{c['why']}]" if c["why"] else ""
            lines.append(f"{indent}{mark} {r['effect']:<6} {r['size']:<6} score {score}{why}")
            if c["status"] in ("kept", "chosen"):
                walk(c, indent + "    ")
    walk(nodes[0], "  ")
    for d, e in enumerate(prepared, start=1):
        lines.append(f"  level {d}: {e['item']['form']} | {e['item']['section']} | filed {e['item']['accepted'][:10]} "
                     f"| novelty {e['novelty']:.2f}")
    return "\n".join(lines)

def find_record(records, words):
    for r in records:
        for e in r["prepared"]:
            if words.lower() in (e["item"]["section"] + " " + e["item"]["text"]).lower():
                return r
    return None

def evidence_check(records):
    """Why a forecast found little evidence: chunks available in its 30-day window and the best score per search."""
    return pd.DataFrame([{"forecast": r["id"], "scenario": r.get("scenario", ""), "items": len(r["prepared"]),
                          **r["search_check"]} for r in records])

def novelty_examples(records, index, n=3):
    """The passages with the lowest and highest novelty, each next to its closest chunk in the previous filing."""
    seen, rows = set(), []
    for r in records:
        for e in r["prepared"]:
            it = e["item"]
            if e["previous_filing"] is None or it["id"] in seen:
                continue
            seen.add(it["id"])
            prev_rows, _ = previous_filing_rows(index, it)
            sims = index.emb[prev_rows] @ index.emb[it["row"]]
            j = prev_rows[int(np.argmax(sims))]
            rows.append({"novelty": round(e["novelty"], 2), "similarity": round(float(np.max(sims)), 3),
                         "passage": f"{it['form']} | {it['section']} | {it['accepted'][:10]}: {it['text'][:300]}",
                         "closest in previous filing": f"{index.chunks[j]['accepted'][:10]}: {index.chunks[j]['text'][:300]}"})
    rows.sort(key=lambda x: x["novelty"])
    return rows[:n], rows[-n:] if len(rows) > n else []

def leakage_audit(records):
    problems = []
    for r in records:
        for e in r["evidence"]:
            if datetime.fromisoformat(e["accepted"]) >= r["as_of"]:
                problems.append((r["id"], e["id"], "evidence on or after the forecast date"))
    return problems

### Test scenarios

The first run only used forecasts made 2 days after results. That is one phase of the quarter (usually a calm month) with one kind of evidence (a results release and a 10-Q, mostly repeated wording). To see where ToT helps and where it does not, forecasts now come from five scenarios:

| Scenario | Forecast date | What it tests |
| --- | --- | --- |
| after results | 2 days after a results filing | the first run's case, for comparison |
| mid-quarter | 30 trading days after the results reaction | quiet periods with little news |
| results coming | 55 trading days after the results reaction, when the next results fall inside the month | the hardest phase for the baseline |
| event | 2 days after an 8-K about a deal, an acquisition, a restructuring, an impairment, a cyber incident or a management change | evidence-rich cases, where ToT should matter most |
| random | Mondays drawn at random | an unbiased check |

Expect few or no evidence items in the mid-quarter and random scenarios: often no company filing falls in the 30 days before the date, and ToT then simply keeps the baseline. News, planned as the next step, is what fills those gaps.

The actual next results date is used only to pick "results coming" dates, never as an input to the forecast. When two scenarios give the same date for a stock, it is counted once, under the first scenario in the order event, after results, results coming, mid-quarter, random.

In [ ]:
SCENARIOS = ("event", "after results", "results coming", "mid-quarter", "random")

def _reaction_index(trading_days, accepted):
    """Index of the first trading day that reflects a filing: the next day if filed after the close."""
    d = np.datetime64(accepted.date(), "D")
    side = "right" if accepted.hour >= 20 else "left"
    return int(np.searchsorted(trading_days, d, side=side))

def _midnight(day):
    d = pd.Timestamp(day).date()
    return datetime(d.year, d.month, d.day, tzinfo=timezone.utc)

def scenario_dates(filings, trading_index, start, end, seed=None):
    """Forecast dates for one stock, each labelled with its scenario."""
    seed = SCENARIO_SEED if seed is None else seed
    lo = datetime.fromisoformat(start).replace(tzinfo=timezone.utc)
    hi = datetime.fromisoformat(end).replace(tzinfo=timezone.utc)
    days = np.array(pd.DatetimeIndex(trading_index).values, dtype="datetime64[D]")
    results = results_dates(filings)
    found = []
    for r in filings:
        items = [c.strip() for c in str(r.get("items", "")).split(",")]
        if r["form"] == "8-K" and "2.02" not in items and any(c in EVENT_ITEMS for c in items):
            acc = parse_accepted(r["acceptanceDateTime"])
            found.append((_midnight(acc) + timedelta(days=DAYS_AFTER_RESULTS), "event"))
    for j, r in enumerate(results):
        found.append((_midnight(r) + timedelta(days=DAYS_AFTER_RESULTS), "after results"))
        k = _reaction_index(days, r)
        if k + MID_QUARTER_DAYS < len(days):
            found.append((_midnight(days[k + MID_QUARTER_DAYS]), "mid-quarter"))
        if k + RESULTS_COMING_DAYS + HORIZON - 1 < len(days) and j + 1 < len(results):
            start_i = k + RESULTS_COMING_DAYS
            nxt = _reaction_index(days, results[j + 1])
            if start_i <= nxt <= start_i + HORIZON - 1:
                found.append((_midnight(days[start_i]), "results coming"))
    mondays = [d for d in pd.date_range(lo, hi, freq="W-MON")]
    rng = np.random.default_rng(seed)
    for d in rng.choice(len(mondays), size=min(RANDOM_PER_TICKER, len(mondays)), replace=False) if mondays else []:
        found.append((_midnight(mondays[d]), "random"))
    best = {}
    for as_of, sc in found:
        if lo <= as_of <= hi and (as_of not in best or SCENARIOS.index(sc) < SCENARIOS.index(best[as_of])):
            best[as_of] = sc
    return sorted(best.items())

def results_by_scenario(records):
    """Mean Brier score per variant within each scenario, plus how much evidence and change there was."""
    rows = []
    for sc in SCENARIOS:
        recs = [r for r in records if r.get("scenario") == sc and r["outcome"]]
        if not recs:
            continue
        row = {"scenario": sc, "forecasts": len(recs),
               "evidence items": round(float(np.mean([len(r["prepared"]) for r in recs])), 1),
               "ToT changed the baseline": sum(bool(r["tot"]["ledger"]) for r in recs)}
        for name in recs[0]["variants"]:
            row[name] = round(float(np.mean([brier(r["variants"][name], r["outcome"]) for r in recs])), 4)
        rows.append(row)
    return pd.DataFrame(rows).set_index("scenario")

### Run the forecasts

About 40 forecasts per stock. With Qwen2.5-3B on a T4 expect 30 to 40 minutes for three stocks. The branch log (every node, pruned ones too) is written to `branch_log.jsonl`.

In [ ]:
import os
if os.path.exists(BRANCH_LOG):
    os.remove(BRANCH_LOG)

RECORDS = []
for t in TICKERS:
    plan = scenario_dates(FILINGS[t], PRICES[t].index, START_DATE, END_DATE)
    print(f"{t}: " + ", ".join(f"{sc} {sum(s == sc for _, s in plan)}" for sc in SCENARIOS))
    for as_of, sc in plan:
        rec = run_forecast(t, as_of, CHAT, INDEX, CASE, PRICES[t], FILINGS[t], embed_query,
                           company=NAMES[t], branch_log=BRANCH_LOG)
        rec["scenario"] = sc
        RECORDS.append(rec)
        print(f"  {rec['id']} [{sc}]: {len(rec['prepared'])} items, {rec['calls_tot']} calls, "
              f"ToT {rec['variants']['ToT beam 3']}  outcome {rec['outcome']}")

def slim(rec):
    keep = {k: v for k, v in rec.items() if k not in ("prepared", "tot", "tot1", "single", "exhaustive")}
    keep["ledger"] = [{k: l[k] for k in ("source", "effect", "size", "reason", "score")} for l in rec["tot"]["ledger"]]
    return keep
json.dump([slim(r) for r in RECORDS], open("records.json", "w"), default=str, indent=1)

### 1. Does searching over readings help?

Lower Brier is better. "vs baseline" is the average difference from the baseline on the same forecasts (negative = better), with a 95% interval from resampling the forecasts. With about 120 forecasts, expect fairly wide intervals: the direction is the signal to look for, not a proof.

In [ ]:
import matplotlib.pyplot as plt

table = results_table(RECORDS)
display(table)
ax = table["mean Brier"].plot(kind="bar", rot=0, figsize=(8, 3.5), color="steelblue",
                              title="Mean Brier score against actual outcomes (lower is better)")
ax.set_ylabel("Brier score")
plt.tight_layout(); plt.show()
print("Prompt-only replies that could be read:", sum(r["prompt_only_parsed"] for r in RECORDS), "of", len(RECORDS))

### By scenario

The same scores within each scenario. With 10 to 25 forecasts per scenario these are indications: look for scenarios where ToT clearly differs from the single pass or the baseline.

In [ ]:
display(results_by_scenario(RECORDS))

### 2. The Apple legal-proceedings case

Checkpoint 3's only approved line was "Apple is involved in legal proceedings that could affect its financial condition", standard wording that the reviewer flagged as weak evidence. Here is how the tree handles a legal-proceedings passage. ★ = chosen, ● = kept in the beam, ○ = pruned (with the reason).

In [ ]:
case_rec = find_record([r for r in RECORDS if r["ticker"] == "AAPL"], "legal proceedings") or RECORDS[0]
print(print_tree(case_rec))
for e in case_rec["prepared"]:
    print(f"\n[{e['item']['section']}] novelty {e['novelty']:.2f}, previous filing {e['previous_filing']}")
    for r in e["readings"]:
        flag = "" if r["ok"] else f"  PRUNED: {r['why']}"
        print(f"  {r['effect']:<6} {r['size']:<6} base rate {r['base_rate']:.2f} (n={r['n_events']}) "
              f"novelty {r['novelty']:.2f} -> {r['score']:.2f}{flag}\n      {r['reason'][:140]}")

### 3. Are beam width 3 and the 0.3 margin sensible?

The class lesson stresses that accuracy depends on the search settings as much as on the model. This reruns the search on the cached readings with other settings, so it costs no model calls.

In [ ]:
display(sweep(RECORDS))

### 4. Search error or scoring error?

- **Beam found the top-scored ledger:** if this is usually true, beam width 3 is wide enough, and a wider beam would not help.
- **A pruned ledger was closer to the outcome:** the Checkpoint 4 risk check. Some of this is luck (one month is one draw); if it happens most of the time, the scoring weights need work.
- **Score vs accuracy:** across all possible ledgers for a forecast, do higher-scored ledgers have lower Brier? Positive is good.

In [ ]:
svs = search_vs_scoring(RECORDS)
display(svs)
print("Beam found the top-scored ledger:", int(svs["beam found the top-scored ledger"].sum()), "of", len(svs))
print("A pruned ledger was closer:", int(svs["a pruned ledger was closer to the outcome"].sum()), "of", len(svs))
print("Average rank correlation:", round(svs["score vs accuracy (rank correlation)"].mean(), 2))

### Diagnostic: novelty and "no effect"

Check the novelty cut-offs (section 9) here. If almost every passage scores near 0 novelty, or ToT keeps the baseline in nearly every forecast, the cut-offs in `NOVELTY_SIM_RANGE` are too strict; if everything scores near 1, too loose.

In [ ]:
nov = pd.Series([e["novelty"] for r in RECORDS for e in r["prepared"] if e["previous_filing"]])
print(nov.describe().round(2))
nov.plot(kind="hist", bins=20, title="Novelty of evidence passages", figsize=(6, 2.5)); plt.show()
kept_baseline = sum(not r["tot"]["ledger"] for r in RECORDS)
print(f"ToT kept the baseline (all 'no effect') in {kept_baseline} of {len(RECORDS)} forecasts")

### Diagnostic: why some forecasts find little evidence

For each forecast: how many chunks of the stock's filings fall in the 30 days before the forecast date, and the best similarity score of each of the three searches (or "nothing relevant found" when even the best is below 0.30). Zero chunks means no filing in the window; chunks but low scores means the searches did not match the filings' wording.

In [ ]:
ec = evidence_check(RECORDS)
display(ec[ec["items"] <= 1])

### Diagnostic: is "repeated wording" really repeated?

The three passages judged least new and the three judged most new, each next to its closest chunk in the previous filing. If the "least new" pairs read as the same text, the novelty check is right. If they read as different news in similar words, the cut-offs in `NOVELTY_SIM_RANGE` are too strict.

In [ ]:
low, high = novelty_examples(RECORDS, INDEX)
for title, group in (("Least new", low), ("Most new", high)):
    print(f"===== {title} =====")
    for ex in group:
        print(f"\nnovelty {ex['novelty']} (similarity {ex['similarity']})")
        print("  THIS:    ", ex["passage"])
        print("  PREVIOUS:", ex["closest in previous filing"])

### 5. Does it fit the budget?

Checkpoint 4 planned about 5 generation calls and up to 15 grounding checks, capped at 25.

In [ ]:
ct = calls_table(RECORDS)
display(ct.describe().loc[["mean", "max"]])

### Leakage audit

Every evidence passage must be dated before its forecast date. Case memory only uses windows that ended before the forecast date by construction (section 8).

In [ ]:
problems = leakage_audit(RECORDS)
print(f"Checked {sum(len(r['evidence']) for r in RECORDS)} evidence passages: {len(problems)} problems")
assert not problems, problems

## 18. What to take from this

Fill in after the run:

1. **Scenarios:** where does ToT differ most from the single pass and the baseline? Event-driven forecasts should show the largest effect.
2. **Single pass vs ToT:** did ToT change ledgers, and in which direction did the Brier score move?
3. **"No effect":** in how many forecasts did ToT keep the baseline? Was the legal-proceedings passage read as "no effect"?
4. **Settings:** did beam width or margin change the chosen ledgers? If not, beam 3 is wide enough.
5. **Search vs scoring:** where errors come from tells you whether to change the search or the weights.
6. **Cost:** calls and seconds per forecast against the Checkpoint 4 budget.

**Limits.** Scored against actual outcomes, not the option-implied answer key; about 120 forecasts in five scenarios; three fixed searches instead of the ReAct loop; default weights; a small open model. The same notebook can be rerun with a 7B or API model by changing `LLM_BACKEND` and `LLM_MODEL`.